Add __pow__ to the Value class so you can compute x ** n. Verify that d/dx(x^3) at x=2 equals 12.0.

In [3]:
class Value:
    def __init__(self,data,children=(),op=''):
        self.data=data
        self.grad=0.0
        self._backward=lambda:None #private/internal — don't touch unless you know what you're doing."
        self._prev=set(children)
        self._op=op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f},grad={self.grad:.4f})"
    
    def __pow__(self, n):
            out = Value(self.data ** n, (self,), f'**{n}')
            def _backward():
                self.grad += n * (self.data ** (n - 1)) * out.grad
            out._backward = _backward
            return out
        
    def backward(self):
        # Topological order all children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Go one variable at a time and apply chain rule
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [4]:
x=Value(2.0)
y=x**3
y.backward()

print(f"x = {x}")
print(f"y = x^3 = {y}")
print(f"dy/dx at x=2: {x.grad}")
print(f"Expected (3*x^2 = 3*4): 12.0")
print(f"Verification: {'PASS' if abs(x.grad - 12.0) < 1e-6 else 'FAIL'}")


x = Value(data=2.0000,grad=12.0000)
y = x^3 = Value(data=8.0000,grad=1.0000)
dy/dx at x=2: 12.0
Expected (3*x^2 = 3*4): 12.0
Verification: PASS


Add tanh as an activation function. Verify that tanh'(0) = 1 and tanh'(2) = 0.0707 (approx).

In [5]:
import math

class Value:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            # d/dx tanh(x) = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ---------- Verification ----------
print("=" * 50)
print("Verifying tanh'(x) = 1 - tanh(x)^2")
print("=" * 50)

# Test 1: tanh'(0) should be 1.0
x1 = Value(0.0)
y1 = x1.tanh()
y1.backward()
print(f"\ntanh'(0):")
print(f"  x.data        = {x1.data}")
print(f"  tanh(x)       = {y1.data}")
print(f"  x.grad        = {x1.grad:.6f}")
print(f"  Expected      = 1.000000")
print(f"  Verification  = {'PASS ✓' if abs(x1.grad - 1.0) < 1e-6 else 'FAIL ✗'}")

# Test 2: tanh'(2) should be approx 0.0707
x2 = Value(2.0)
y2 = x2.tanh()
y2.backward()
print(f"\ntanh'(2):")
print(f"  x.data        = {x2.data}")
print(f"  tanh(2)       = {y2.data:.6f}")
print(f"  x.grad        = {x2.grad:.6f}")
print(f"  Expected      = 0.070650")
print(f"  Verification  = {'PASS ✓' if abs(x2.grad - 0.07065) < 1e-4 else 'FAIL ✗'}")

Verifying tanh'(x) = 1 - tanh(x)^2

tanh'(0):
  x.data        = 0.0
  tanh(x)       = 0.0
  x.grad        = 1.000000
  Expected      = 1.000000
  Verification  = PASS ✓

tanh'(2):
  x.data        = 2.0
  tanh(2)       = 0.964028
  x.grad        = 0.070651
  Expected      = 0.070650
  Verification  = PASS ✓
